# 🧱 Modul 09: Hyperparameter Tuning, Pipeline & Serialisasi Model
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/09_Pipeline_and_Model_Export.ipynb)
[![Course](https://img.shields.io/badge/Course-Data%20Mining%20Zero%20to%20Hero-blue.svg)](https://github.com/antonprafanto/data-mining-zero-to-hero)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

> **Pengampu**: Anton Prafanto  
> **Prinsip Pembelajaran**: *Intuition before Formulas* — Pahami filosofi & analogi dunia nyata sebelum rumus matematika dan kode.

---

### 🗺️ Peta Posisi Modul dalam CRISP-DM
```
[1. Business Understanding] ➔ [2. Data Understanding] ➔ [3. Data Preparation]
                                                               │
                                                               ▼
[6. Deployment] 💡 ◄── 🎯 [5. EVALUATION & TUNING] ◄── [4. Modeling]
```
Banyak pemula merasa pekerjaannya selesai saat model menghasilkan akurasi tinggi di dalam sel notebook. Padahal, **model yang terkunci di dalam memori notebook tidak bisa digunakan oleh siapa pun di dunia nyata**!  
Di **Modul 09**, kita bertransformasi menjadi **Machine Learning Engineer Profesional**: Mengoptimalkan performa model dengan *Hyperparameter Tuning*, menyatukan rantai prapemrosesan ke dalam *Scikit-Learn Pipeline*, dan mengekspor model menjadi file biner (`.joblib`) yang siap dipasang di aplikasi web!


## 🏭 1. Filosofi Menuju Level Profesional: Mengapa Notebook Saja Tidak Cukup?

### 🍳 Analogi Dapur Restoran vs Pabrik Makanan Siap Saji
* **Notebook (Dapur Manual)**: Di sel 1 Anda mengisi nilai hilang, di sel 2 Anda melakukan one-hot encoding, di sel 3 Anda scaling, lalu di sel 4 Anda melatih model. Jika ada pesanan baru, Anda harus mengulang seluruh langkah manual tersebut. Ini rentan salah langkah (*human error*) dan kebocoran informasi (*Data Leakage*).
* **Pipeline (Lini Pabrik Otomatis)**: Kita merakit sebuah ban berjalan (*assembly line*). Data mentah kotor masuk di ujung kiri, otomatis dibersihkan, di-skala, dan di ujung kanan langsung keluar prediksi instan!

---

### 📦 3 Keuntungan Mutlak Scikit-Learn Pipeline:
1. **Bebas Data Leakage**: Pipeline menjamin operasi *fit* hanya dilakukan pada data latih, dan *transform* otomatis diterapkan pada data uji atau data produksi baru.
2. **Kode Bersih & Rapi**: Cukup satu perintah `pipeline.fit()` dan `pipeline.predict()`.
3. **Mudah Dipindahkan (*Portable*)**: Cukup simpan satu file `model.joblib`, dan seluruh rantai transformasi ikut tersimpan di dalamnya!


## 🔄 2. Validasi Silang (*K-Fold Cross-Validation*): Menghindari Bias Ujian Acak

### 🎓 Analogi Ujian Bergilir
Jika kita hanya membagi data 1 kali (`train_test_split`), model kita bisa saja kebetulan mendapatkan soal ujian yang sangat gampang (sehingga akurasi menipu tinggi) atau kebetulan soalnya sangat aneh (sehingga akurasi anjlok).

**K-Fold Cross-Validation (K=5)** membagi data latih menjadi 5 bagian (*folds*):
* **Putaran 1**: Fold 1, 2, 3, 4 dipakai belajar ➔ Fold 5 dipakai ujian.
* **Putaran 2**: Fold 1, 2, 3, 5 dipakai belajar ➔ Fold 4 dipakai ujian.
* ...dan seterusnya bergilir sampai 5 putaran!
* Nilai akhir adalah **rata-rata skor dari ke-5 putaran**, memberikan estimasi performa model yang jauh lebih objektif dan tahan banting.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestClassifier

# Memuat dataset
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

# Inisialisasi K-Fold (5 Lipatan)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
rf_baseline = RandomForestClassifier(n_estimators=50, random_state=42)

# Hitung Skor Validasi Silang
skor_cv = cross_val_score(rf_baseline, X, y, cv=kfold, scoring='accuracy')

print("=== HASIL 5-FOLD CROSS VALIDATION ===")
for i, skor in enumerate(skor_cv, 1):
    print(f"Putaran {i} : Akurasi {skor*100:.2f}%")
    
print(f"\n🎯 Rata-rata Akurasi (Mean CV Score) : {skor_cv.mean()*100:.2f}%")
print(f"📊 Standar Deviasi Skor              : ±{skor_cv.std()*100:.2f}% (Semakin kecil semakin stabil!)")


## 🎛️ 3. Penyetelan Hyperparameter: GridSearchCV vs RandomizedSearchCV

### 🎚️ Apa Bedanya Parameter vs Hyperparameter?
* **Parameter**: Nilai internal yang dipelajari dan disesuaikan sendiri oleh model dari data (contoh: koefisien kemiringan $\beta$ pada regresi, bobot percabangan pohon).
* **Hyperparameter**: "Tombol setelan di luar kabin" yang harus kita atur sebelum model mulai belajar (contoh: `n_estimators`, `max_depth`, `min_samples_split`, `learning_rate`).

---

### 🔍 Dua Pendekatan Pencarian Parameter Optimal:
1. **GridSearchCV (Menyeluruh / Kisi Lengkap)**: Mencoba *seluruh kemungkinan kombinasi* parameter layaknya tabel perkalian. Dijamin menemukan kombinasi terbaik di dalam kisi, namun membutuhkan waktu komputasi yang lama jika kombinasinya ribuan.
2. **RandomizedSearchCV (Acak Cerdas)**: Mengambil sampel kombinasi secara acak dalam batas iterasi tertentu. Jauh lebih cepat (bisa 10x lebih hemat waktu) dengan hasil performa yang sering kali setara!


In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Menentukan ruang hyperparameter yang ingin diuji
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 8, None],
    'min_samples_split': [2, 5, 10]
}

# Total kombinasi: 3 x 4 x 3 = 36 kombinasi! (Dengan 5-fold CV = 180 kali pelatihan!)
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X, y)

print("=== HASIL GRIDSEARCHCV ===")
print(f"🏆 Akurasi Terbaik (Best CV Score): {grid_search.best_score_*100:.2f}%")
print("🔧 Kombinasi Parameter Juara:")
for param, val in grid_search.best_params_.items():
    print(f" - {param:<18}: {val}")


## 🔗 4. Membangun Pipeline Lengkap (*Scikit-Learn ColumnTransformer*)

Sekarang mari kita bangun **Pipeline Produksi Skala Penuh** menggunakan dataset Titanic (yang memuat campuran data numerik, kategorikal, dan missing values) sehingga model kita siap menerima data kotor apa pun dari dunia luar secara aman!


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Data Simulasi Penumpang Kapal (Titanic-like Data)
df_titanic = pd.DataFrame({
    'Umur': [22, 38, 26, 35, np.nan, 54, 2, 27, np.nan, 40],
    'Tarif_Tiket': [7.25, 71.83, 7.92, 53.10, 8.05, 51.86, 21.07, 11.13, 30.07, 16.70],
    'Jenis_Kelamin': ['pria', 'wanita', 'wanita', 'wanita', 'pria', 'pria', 'pria', 'wanita', 'wanita', 'pria'],
    'Kelas_Kabin': ['Ekonomi', 'VIP', 'Ekonomi', 'VIP', 'Ekonomi', 'VIP', 'Bisnis', 'Bisnis', 'VIP', 'Bisnis'],
    'Selamat': [0, 1, 1, 1, 0, 0, 1, 1, 1, 0] # Target (0 = Tidak, 1 = Selamat)
})

X_raw = df_titanic[['Umur', 'Tarif_Tiket', 'Jenis_Kelamin', 'Kelas_Kabin']]
y_raw = df_titanic['Selamat']

# Tentukan kelompok kolom
fitur_numerik = ['Umur', 'Tarif_Tiket']
fitur_kategori = ['Jenis_Kelamin', 'Kelas_Kabin']

# 1. Pipa Jalur Numerik: Imputasi Median -> Standarisasi
pipa_numerik = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Pipa Jalur Kategori: Imputasi Modus -> One-Hot Encoding
pipa_kategori = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# 3. Penggabung Jalur: ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', pipa_numerik, fitur_numerik),
    ('cat', pipa_kategori, fitur_kategori)
])

# 4. Pipeline Final Terpadu: Preprocessing + Model Klasifikasi
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42))
])

# Latih seluruh rantai pipa hanya dengan SATU BARIS PERINTAH!
model_pipeline.fit(X_raw, y_raw)
print("✅ Seluruh rantai Pipeline berhasil dirakit dan dilatih dengan sempurna!")


## 💾 5. Serialisasi Model: Menyimpan & Memuat Kembali dengan `joblib`

Model yang sudah dilatih disimpan ke file biner (`.joblib`). File ini menyimpan seluruh memori bobot model, kamus OneHotEncoder, serta nilai median/mean untuk imputasi. Siapa pun yang memiliki file `.joblib` ini dapat langsung memakainya di server web tanpa perlu melatih ulang!


In [ ]:
import joblib

nama_file_model = "model_prediksi_titanic_pipeline.joblib"

# 1. Simpan Pipeline ke Disk
joblib.dump(model_pipeline, nama_file_model)
print(f"💾 File berhasil disimpan ke disk: {nama_file_model}")

# 2. Muat Ulang Pipeline dari Disk (Mensimulasikan lingkungan server aplikasi baru)
pipeline_termuat = joblib.load(nama_file_model)
print("🔄 File berhasil dimuat kembali ke memori!")

# 3. Uji Prediksi Penumpang Baru (Data Mentah dengan Missing Values!)
penumpang_baru = pd.DataFrame([{
    'Umur': np.nan,            # Nilai hilang (akan otomatis diimputasi median oleh pipeline)
    'Tarif_Tiket': 65.50,
    'Jenis_Kelamin': 'wanita',
    'Kelas_Kabin': 'VIP'
}])

hasil_prediksi = pipeline_termuat.predict(penumpang_baru)[0]
peluang_selamat = pipeline_termuat.predict_proba(penumpang_baru)[0][1]

status_teks = "SELAMAT 🌟" if hasil_prediksi == 1 else "TIDAK SELAMAT ⚠️"
print("\n=== HASIL INFERENSI DATA PENUMPANG BARU ===")
print(f"Data Masukan    : Wanita, Tiket VIP ($65.50), Umur Tidak Diketahui")
print(f"Status Prediksi : {status_teks}")
print(f"Peluang Selamat : {peluang_selamat*100:.1f}%")


## 🧠 6. Kuis Evaluasi Pemahaman Mandiri

Uji pemahaman Anda terhadap konsep pipeline dan rekayasa model, lalu jalankan sel kode di bawah untuk memeriksa kunci jawabannya:

---
* **Soal 1**: Mengapa K-Fold Cross-Validation jauh lebih dianjurkan daripada pembagian data acak tunggal (*single train-test split*) saat membandingkan performa beberapa algoritma?
* **Soal 2**: Apa perbedaan mendasar antara *Parameter* dan *Hyperparameter* pada model machine learning?
* **Soal 3**: Mengapa penggunaan `ColumnTransformer` dan `Pipeline` dari Scikit-Learn mutlak diperlukan untuk mencegah *Data Leakage*?
* **Soal 4**: Mengapa kita menyimpan seluruh objek `Pipeline` ke dalam `.joblib`, bukan hanya menyimpan objek model `RandomForestClassifier`-nya saja?


In [ ]:
# Kunci jawaban logis evaluasi mandiri Modul 9
kunci_jawaban_modul_9 = {
    "Soal 1": "Karena K-Fold menguji model di seluruh bagian data secara bergilir (5 atau 10 kali) dan menghitung skor rata-rata, sehingga performa yang dihasilkan tidak bergantung pada keberuntungan atau ketidakberuntungan pembagian acak tunggal.",
    "Soal 2": "Parameter adalah bobot internal yang dipelajari mesin sendiri dari data latih, sedangkan Hyperparameter adalah setelan konfigurasi eksternal yang ditentukan manusia sebelum pelatihan dimulai.",
    "Soal 3": "Karena Pipeline memastikan transformasi data (seperti mean imputer dan standard deviation scaler) hanya dihitung dari data latih pada setiap fold dan diterapkan secara terisolasi tanpa bocor ke data uji.",
    "Soal 4": "Jika hanya menyimpan modelnya saja, aplikasi web di masa depan akan gagal memproses data mentah pengguna yang belum di-imputasi, di-scale, dan di-one-hot encoding. Menyimpan seluruh Pipeline menjamin data mentah pengguna dapat langsung diproses otomatis tanpa kode transformasi tambahan."
}

print("=== KUNCI JAWABAN EVALUASI MANDIRI MODUL 09 ===")
for soal, pembahasan in kunci_jawaban_modul_9.items():
    print(f"👉 {soal}:\n   {pembahasan}\n")


## ✍️ 7. Kotak Latihan Bebas (*Playground Challenge*)

### 🎯 Misi Eksperimen: Menguji Penumpang Laki-Laki Kelas Ekonomi
Ujilah model pipeline yang telah disimpan tadi untuk seorang penumpang baru:
* `Umur` = 25 tahun
* `Tarif_Tiket` = 7.50 USD
* `Jenis_Kelamin` = 'pria'
* `Kelas_Kabin` = 'Ekonomi'

Tugas Anda:
Jalankan inferensi dan cari tahu berapa persen peluang selamat penumpang tersebut menurut model AI!


In [ ]:
# Solusi Tantangan Mandiri
penumpang_latihan = pd.DataFrame([{
    'Umur': 25.0,
    'Tarif_Tiket': 7.50,
    'Jenis_Kelamin': 'pria',
    'Kelas_Kabin': 'Ekonomi'
}])

pred_latihan = pipeline_termuat.predict(penumpang_latihan)[0]
prob_latihan = pipeline_termuat.predict_proba(penumpang_latihan)[0]

status_latihan = "SELAMAT 🌟" if pred_latihan == 1 else "TIDAK SELAMAT ⚠️"
print(f"Penumpang: Pria 25 tahun, Tiket Ekonomi ($7.50)")
print(f"Prediksi AI: {status_latihan} (Peluang Selamat: {prob_latihan[1]*100:.1f}%, Risiko Tidak Selamat: {prob_latihan[0]*100:.1f}%)")


## 🎯 8. Checklist Kelulusan Modul 09 (*Hero Checkpoint*)

Beri tanda centang pada hati Anda jika telah menguasai indikator penting berikut:
- [ ] Saya memahami mengapa notebook manual tidak cukup untuk kebutuhan produksi aplikasi nyata.
- [ ] Saya menguasai konsep dan cara kerja **K-Fold Cross-Validation** dalam menguji stabilitas model.
- [ ] Saya dapat membedakan konsep Parameter vs Hyperparameter, serta cara kerja **GridSearchCV**.
- [ ] Saya mampu merakit **ColumnTransformer** dan **Pipeline** untuk memproses fitur numerik dan kategorikal sekaligus.
- [ ] Saya berhasil mengekspor seluruh pipeline ke file biner `.joblib` dan memuatnya kembali untuk inferensi data baru.

---
🚀 **Selamat! Anda telah siap menjadi Fullstack Data Practitioner!**  
Di modul berikutnya (**Modul 10**), kita akan memasang file model `.joblib` ini ke dalam aplikasi web interaktif nyata menggunakan **Streamlit**, lalu menjalankannya langsung dari Google Colab hingga deploy gratis ke cloud!
